In [0]:
from pyspark.sql import functions as F, Window

silver_path = "s3://enterprise-lakehouse-data/silver/asset_snapshots/"
curated_path = "s3://enterprise-lakehouse-data/silver_curated/assets_scd/"

silver_asset_df = spark.read.parquet(silver_path)


In [0]:
silver_asset_df.count()

16873

Define business-effective time


In [0]:
base_df = (
    silver_asset_df
    .withColumn(
        "effective_from",
        F.coalesce(F.col("last_updated_dt"), F.col("ingestion_ts"))
    )
)


Collapse to ONE Row per asset_id

In [0]:
w = Window.partitionBy("asset_id").orderBy(F.col("effective_from").desc())

latest_asset_df = (
    base_df
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)



Add SCD columns

In [0]:
assets_scd_bootstrap_df = (
    latest_asset_df
    .withColumn("effective_to", F.lit(None).cast("date"))
    .withColumn("is_current", F.lit(True))
)


Overwrite curated table (ALLOWED — still bootstrap)

In [0]:
(
    assets_scd_bootstrap_df
    .write
    .mode("overwrite")   # REQUIRED for bootstrap
    .parquet(curated_path)
)


Validation

In [0]:
spark.read.parquet(curated_path) \
     .groupBy("asset_id") \
     .count() \
     .filter("count > 1") \
     .show()


+--------+-----+
|asset_id|count|
+--------+-----+
+--------+-----+



In [0]:
silver_asset_df.select("asset_id").distinct().count()


201

In [0]:
spark.read.parquet(curated_path).count()


201

Build FULL SCD Type-2 History (Pure Spark / Window-based)

In [0]:
from pyspark.sql import functions as F, Window

silver_path = "s3://enterprise-lakehouse-data/silver/asset_snapshots/"
curated_path = "s3://enterprise-lakehouse-data/silver_curated/assets_scd/"

silver_df = spark.read.parquet(silver_path)


Define Business Effective Time (MANDATORY)

In [0]:
df = (
    silver_df
    .withColumn(
        "effective_from",
        F.coalesce(F.col("last_updated_dt"), F.col("ingestion_ts"))
    )
)


Order Snapshots Per Asset


In [0]:
w_order = Window.partitionBy("asset_id").orderBy("effective_from")


Detect Changes Using HASH (CORE LOGIC)

In [0]:
df_with_prev = (
    df
    .withColumn("prev_hash", F.lag("asset_event_hash").over(w_order))
)


Keep Only Meaningful Versions

In [0]:
scd_changes_df = (
    df_with_prev
    .filter(
        F.col("prev_hash").isNull() |
        (F.col("asset_event_hash") != F.col("prev_hash"))
    )
    .drop("prev_hash")
)


Calculate effective_to

In [0]:
w_next = Window.partitionBy("asset_id").orderBy("effective_from")

scd_with_end = (
    scd_changes_df
    .withColumn(
        "effective_to",
        F.lead("effective_from").over(w_next)
    )
)


Set is_current

In [0]:
final_scd_df = (
    scd_with_end
    .withColumn(
        "is_current",
        F.when(F.col("effective_to").isNull(), F.lit(True))
         .otherwise(F.lit(False))
    )
)


WRITE — OVERWRITE Curated (YES, AGAIN)

In [0]:
(
    final_scd_df
    .write
    .mode("overwrite")   # CORRECT for history build
    .parquet(curated_path)
)


In [0]:
spark.read.parquet(curated_path) \
    .filter("is_current = true") \
    .groupBy("asset_id") \
    .count() \
    .filter("count > 1") \
    .show()


+--------+-----+
|asset_id|count|
+--------+-----+
+--------+-----+



In [0]:
spark.read.parquet(curated_path) \
    .filter("is_current = true") \
    .count()


201

In [0]:
spark.read.parquet(curated_path) \
    .groupBy("asset_id") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(5)


+--------+-----+
|asset_id|count|
+--------+-----+
|AST-1128|  113|
|AST-1198|  110|
|AST-1030|  108|
|AST-1045|  107|
|AST-1054|  107|
+--------+-----+
only showing top 5 rows


In [0]:
df=spark.read.parquet(curated_path)

In [0]:
df.show()

+--------+--------+----------+------------+-----------+--------+------------+--------------------+---------------+---------------+--------------------+---------------+-----------------+-------------+------------------+-------------------+--------------+--------------------+--------------+-------------------+-------------------+----------+
|asset_id|plant_id|asset_type|install_date|capacity_mw|  status|last_updated| ingestion_timestamp|install_date_dt|last_updated_dt|        ingestion_ts|f_null_asset_id|f_null_asset_type|f_null_status|f_invalid_capacity|f_null_ingestion_ts|failure_reason|    asset_event_hash|ingestion_date|     effective_from|       effective_to|is_current|
+--------+--------+----------+------------+-----------+--------+------------+--------------------+---------------+---------------+--------------------+---------------+-----------------+-------------+------------------+-------------------+--------------+--------------------+--------------+-------------------+---------

In [0]:
spark.read.parquet(curated_path) \
    .filter("is_current = true") \
    .groupBy("asset_id") \
    .count() \
    .filter("count > 1") \
    .show()


+--------+-----+
|asset_id|count|
+--------+-----+
+--------+-----+



In [0]:
spark.read.parquet(curated_path) \
    .filter("effective_to < effective_from") \
    .count()


0

REPLAY SAFETY PROOF

Capture Baseline Metrics

In [0]:
curated_df = spark.read.parquet(curated_path)

baseline_metrics = {
    "total_rows": curated_df.count(),
    "current_rows": curated_df.filter("is_current = true").count(),
    "distinct_assets": curated_df.select("asset_id").distinct().count(),
    "max_effective_from": curated_df.agg({"effective_from": "max"}).collect()[0][0]
}

baseline_metrics


{'total_rows': 16873,
 'current_rows': 201,
 'distinct_assets': 201,
 'max_effective_from': datetime.datetime(2026, 11, 3, 0, 0)}

Capture Metrics AGAIN

In [0]:
after_df = spark.read.parquet(curated_path)

after_metrics = {
    "total_rows": after_df.count(),
    "current_rows": after_df.filter("is_current = true").count(),
    "distinct_assets": after_df.select("asset_id").distinct().count(),
    "max_effective_from": after_df.agg({"effective_from": "max"}).collect()[0][0]
}

after_metrics


{'total_rows': 16873,
 'current_rows': 201,
 'distinct_assets': 201,
 'max_effective_from': datetime.datetime(2026, 11, 3, 0, 0)}

LATE DATA / CORRECTION HANDLING

In [0]:
curated_df = spark.read.parquet(curated_path)

curated_df \
    .filter("asset_id = 'AST-1000'") \
    .orderBy("effective_from") \
    .show(truncate=False)


+--------+--------+----------+------------+-----------+--------+------------+---------------------------+---------------+---------------+--------------------------+---------------+-----------------+-------------+------------------+-------------------+--------------+----------------------------------------------------------------+--------------+-------------------+-------------------+----------+
|asset_id|plant_id|asset_type|install_date|capacity_mw|status  |last_updated|ingestion_timestamp        |install_date_dt|last_updated_dt|ingestion_ts              |f_null_asset_id|f_null_asset_type|f_null_status|f_invalid_capacity|f_null_ingestion_ts|failure_reason|asset_event_hash                                                |ingestion_date|effective_from     |effective_to       |is_current|
+--------+--------+----------+------------+-----------+--------+------------+---------------------------+---------------+---------------+--------------------------+---------------+-----------------+------

Create a Late Correction (Silver-Side Simulation)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

late_correction_df = (
    silver_df
    .filter("asset_id = 'AST-1000'")
    .orderBy("last_updated_dt")
    .limit(1)
    .withColumn("capacity_mw", F.col("capacity_mw") + 10)
    .withColumn("ingestion_ts", current_timestamp())
)


In [0]:
(
    late_correction_df
    .write
    .mode("append")
    .parquet(silver_path)
)


In [0]:
spark.read.parquet(curated_path) \
    .filter("asset_id = 'AST-1000' AND is_current = true") \
    .show(truncate=False)


+--------+--------+----------+------------+-----------+------+------------+---------------------------+---------------+---------------+--------------------------+---------------+-----------------+-------------+------------------+-------------------+--------------+----------------------------------------------------------------+--------------+-------------------+------------+----------+
|asset_id|plant_id|asset_type|install_date|capacity_mw|status|last_updated|ingestion_timestamp        |install_date_dt|last_updated_dt|ingestion_ts              |f_null_asset_id|f_null_asset_type|f_null_status|f_invalid_capacity|f_null_ingestion_ts|failure_reason|asset_event_hash                                                |ingestion_date|effective_from     |effective_to|is_current|
+--------+--------+----------+------------+-----------+------+------------+---------------------------+---------------+---------------+--------------------------+---------------+-----------------+-------------+------------